# Lifetime Risk of Developing Disease

Didn’t found well explained resources on this topic. Below is my attempt to make one while learning. Disclaimer: Notes may be incomplete/contain mistakes.

In [ ]:
knitr::opts_chunk$set(echo = TRUE)

## Goal

Given a cross-sectional dataset (a slice in time) with incidence rate (disease count/total population). we want to estimate the “lifetime risk”, a cumulative probability for a person to get this disease throughout their lifespan.

With this measure, the motivation to supplement health professionals with evidence on managing disease X, answering questions such as:

1.  Burden of cancer on healthcare system? How many patients expected to be treated per year?
2.  What age is the best for cancer screening?
3.  Does different cancer type have diffent optimal age for screening?
4.  Is the cancer screening effective at earlier cancer diagnostic/reach enough people?
5.  Does cancer happen in younger age than before?

## Issues

> **Misconception**
>
> With cross sectional dataset of 100 people and 10 new cases of cancer. Probability of newborn getting cancer is 10%..

### Issue 1: People May Develop Cancer Later

Population of study might still get cancer in later ages. Some are still children.

**Counter: Filter population of study to be people who died.**

1.  We would ignore everyone who is alive, lose a lot of information.
2.  Oudated data since those who died might developed cancer decades ago.

**Counter: To not waste information**

In theory we could, just use the data up to certain age. For example if partient A is current 50 years old and had cancer previously. This would increase the rate of cancer by 50 year old. However, the difficutly is the assumption that we have patients record throughout their whole life (Longitudinal data), which usually might not be complete or is expensive/difficutly to get for millions/billions of population.

### Issue 2: Death Is a Competing Risk

Death is a “competing risk” (i.e. Newborn may die before getting cancer.)

> **Caution**
>
> An extreme example: If all infant vaccines are suddenly ineffective, most of our newborn in our test case likely die before reaching old enough age for (10% rate) cancer to be observed.

## Methods

### Accounting for Age Groups

@tip-1 motivates us to make a rate by age groups. For the toy example

| Age | Population | New Cancer Cases | Death | Rate | Deaths |
|:---:|:----------:|:----------------:|:-----:|:----:|:------:|
|  1  |    100     |        60        |  20   | 0.6  |  0.2   |
|  2  |    100     |        50        |  10   | 0.5  |  0.1   |

> **The 100 people of age 1 and age 2 in our cross-sectional dataset represent different groups of people.**
>
> $$\begin{aligned}
> \mathbb{P}(\text{cancer}) &\neq \mathbb{P}(\text{cancer } Y_1) + \mathbb{P}(\text{cancer } Y_2) \\
> &= 0.6 + 0.5 \\
> &= 1.1
> \end{aligned}$$

A person can only experience cancer in age 2 only if death didnt occur in age 1. Hence, we need conditional probability. The idea is to imagine applying probability on a hypothetical newborn cohort:

flowchart LR
 A(100 newborn) --> B1(60 Cancer)
 A --> B2(20 Cancer-free)
 A --> B3(20 Death)
 B2 --> C1(10 Cancer)
 B2 --> C2(8 Cancer)
 B2 --> C3(2 Cancer)

> **Misconception**
>
> $$\begin{aligned}
> \mathbb{P}(C) &\neq \mathbb{P}(C_{Y_1}) + \mathbb{P}(C_{Y_2} \cap \text{alive in } Y_1) \\
> &= 0.6 + \mathbb{P}(C_{Y_2} \mid \text{alive}) \cdot \mathbb{P}(\text{alive in } Y_1) \\
> &= 0.6 + (0.5) \cdot (1 - 0.2) \\
> &= 1
> \end{aligned}$$

> **Misconception**
>
> $$\begin{aligned}
> \mathbb{P}(C) &= \mathbb{P}(C_{Y_1}) + \mathbb{P}(C_{Y_2} \cap \text{alive in } Y_1 \cap C_{\text{not in } Y_1})
> \\
> &= 0.6 + \mathbb{P}\left[C_{Y_2} \mid C_{\text{not in } Y_1} \cap \text{alive}\right] \cdot \mathbb{P}\underbrace{\left[C_{\text{not in } Y_1} \cap \text{alive}\right]}_{\text{alive cancer-free in } Y_2}
> \\
> &= 0.6 + \frac{\text{cancer}}{\text{cancer} + \text{death}} \times \text{cancer in }  Y_2 \text{ or death} \times \text{Survive } Y_1
> \\
> &\neq? 0.6 + \left[\frac{0.5}{0.5 + 0.1} \times (0.5 + 0.1)\right] \times \left[1 - (0.6 + 0.2)\right]
> \end{aligned}$$

But $0.5$ is not incidence rate of cancer in $Y_2$ because the denominator (100) in $\frac{50}{100}$ is not population at risk. It may include people who had cancer in Age 1? <!-- maybe i confused the rate to probabillity? but hmm pop at risk still not 100 though -->

### Converting Incidence Rate to Probability

In the methods above, we assumed incidence rate = probability (0,5 incidence rate = 50% chance of getting cancer). This needs corrections, arising from the fact that our crosss-sectional dataset compiles the information at a specific interval. We only see one summary slice of a completed picture. In reality, we assume that as age increases every small amount of time, risk of cancer also varies.

Also, once a person develop cancer, they are removed from the population at risk, hence probability might increase given a fixed incidence rate within the interval. For our example, a person can develop cancer at t=0.1 even if our records is compiled only at age=1.

Another concern: A car drove for 100km in one hour has rate of 100km/h. That doesn’t mean the car drove the same speed all the way, it can slow down/speed up along the way.

To account for the length of time considered and how the rate behaves over that period, under the simplifying assumption of a constant rate, developing cancer within time (t) is exponential distribution $f(t) = re^{-rt}$. The probability of developing cancer within time (t) is $\int_0^t re^{-ru}\,du = 1-e^{-rt}$, rather than simply $rt$.

But why exp() though? Can be $2^{rt}$?

Or is it because of other reasons: memoryless property of exponential distribution? connection to poisson/survival models? For now, we think of it as a correction from discrete time to continuous.

The derivation: $$\mathbb{P}(\text{cancer}) \approx r\Delta t$$ $$\mathbb{P}(\text{no cancer}) \approx 1-r\Delta t\\$$ $$\left(1-\frac{r}{n}\right)^n \to e^{-r} \quad \text{as } n\to\infty$$

| Time (t) (years) | Approximation $rt$ | Exact probability $(1-e^{-rt}\)$ |
|-----------------:|-------------------:|---------------------------------:|
|             0.01 |              0.50% |                            0.50% |
|              0.1 |              5.00% |                            4.88% |
|              0.5 |             25.00% |                           22.12% |
|              1.0 |             50.00% |                           39.35% |

> **Converting Rate to Probability, @tip-2 becomes:**
>
> Let $\lambda_{c,j}$ be the cancer rate and $\lambda_{d,j}$ the death rate in $Y_j$, with $\Lambda_j = \lambda_{c,j} + \lambda_{d,j}$.
>
> $$\begin{aligned}
> \mathbb{P}(C) &= \mathbb{P}(C_{Y_1}) + \mathbb{P}(C_{Y_2} \cap \text{alive in } Y_1 \cap C_{\text{not in } Y_1}) \\
> &= \frac{\lambda_{c,1}}{\Lambda_1}\left(1 - e^{-\Lambda_1}\right) + \underbrace{\frac{\lambda_{c,2}}{\Lambda_2}}_{\text{cancer share}} \times \underbrace{\left(1 - e^{-\Lambda_2}\right)}_{\text{cancer or death in } Y_2} \times \underbrace{e^{-\Lambda_1}}_{\text{survive } Y_1} \\
> &= \frac{0.6}{0.8}\left(1 - e^{-0.8}\right) + \frac{0.5}{0.6} \times \left(1 - e^{-0.6}\right) \times e^{-0.8} \\
> &\approx 0.413 + 0.169 = 0.582
> \end{aligned}$$

### Generalised to AMP Method. Extending to more than 2 age groups

Let $q_j = \dfrac{\lambda_{c,j}}{\Lambda_j}\left(1 - e^{-\Lambda_j}\right)$ be the probability of cancer in $Y_j$, given event-free at the start of $Y_j$.

Three age groups, written out in full:
$$\mathbb{P}(C) = q_1 + q_2\, e^{-\Lambda_1} + q_3\, e^{-\Lambda_1}\, e^{-\Lambda_2}$$

$J$ age groups, collapsing the product of exponentials into one exponential of a sum:

> **Generalised with extended time**
>
> $$\begin{aligned}
> \mathbb{P}(C) &= q_1 + q_2\, e^{-\Lambda_1} + q_3\, e^{-\Lambda_1} e^{-\Lambda_2} + \dots + q_J \prod_{k=1}^{J-1} e^{-\Lambda_k}
> \\
> &= \sum_{j=1}^{J} q_j \prod_{k=1}^{j-1} e^{-\Lambda_k}
> \\
> &= \sum_{j=1}^{J} q_j \exp\left(-\sum_{k=1}^{j-1} \Lambda_k\right)
> \\
> &= \sum_{j=1}^{J} \frac{\lambda_{c,j}}{\Lambda_j}\left(1 - e^{-\Lambda_j}\right) \exp\left(-\sum_{k=1}^{j-1} \Lambda_k\right)
> \end{aligned}$$

Note: To avoid overcounting (patient develop cancer and die), our death rate should be solely non-cancer death.

## Simulation/Codes

## Concerns/Limitation

Througout the previous sections, we assume we had new cases of cancer. But in reality we usually don’t?